⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Migration: ODI Session Conversion\n",
    "**Source File:** target_sql_file.sql  \n",
    "**Target Environment:** Databricks Delta Lake  \n",
    "**Conversion Date:** 2024-05-23"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
    "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
    "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### ETL Parameters"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "display(spark.sql(f\"\"\"\n",
    "SELECT \n",
    "    '{dbutils.widgets.get(\"DATASOURCE_NUM_ID\")}' AS datasource_num_id,\n",
    "    '{dbutils.widgets.get(\"ETL_PROC_WID\")}' AS etl_proc_wid,\n",
    "    '{dbutils.widgets.get(\"ODI_SESS_NO\")}' AS odi_sess_no\n",
    "\"\"\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### File Capture Logic (SCEN_TASK_NO 1)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ODI OdiOSCommand converted to dbutils.fs.ls and data processing\n",
    "# Logic: Listing employee directory and capturing filenames\n",
    "\n",
    "import pandas as pd\n",
    "\n",
    "try:\n",
    "    # Replace local path with Databricks File System (DBFS) path\n",
    "    files = dbutils.fs.ls(\"/mnt/odi/employee\")\n",
    "    file_names = [f.name for f in files if not f.isDir()]\n",
    "    \n",
    "    # Creating the name.csv equivalent\n",
    "    df = pd.DataFrame(file_names, columns=[\"file_name\"])\n",
    "    # Logic to save to target directory\n",
    "    # df.to_csv(\"/dbfs/mnt/odi/employeesfile/name.csv\", index=False, header=False)\n",
    "    print(f\"Captured {len(file_names)} files.\")\n",
    "except Exception as e:\n",
    "    print(f\"Error capturing file list: {e}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Truncate Log Table (SCEN_TASK_NO 2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "TRUNCATE TABLE workspace.odi_trg.log_table;"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Sequence Management (SCEN_TASK_NO 3)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Oracle ALTER_SEQUENCE restart logic converted to Delta Identity Restart\n",
    "-- Note: Identify the primary table utilizing 'file_sequence' and restart its identity column\n",
    "-- Example assuming log_table uses this sequence for a column named 'id'\n",
    "\n",
    "ALTER TABLE workspace.odi_trg.log_table ALTER COLUMN id RESTART WITH 1;"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Validation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "SELECT COUNT(*) as log_count FROM workspace.odi_trg.log_table;"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Conversion Notes\n",
    "1. **OS Command:** The Windows `dir` command was converted to a Python cell using `dbutils.fs.ls`. Local C:\\\ paths must be mapped to DBFS or external storage mounts.\n",
    "2. **Truncate:** Straightforward mapping to `workspace.odi_trg.log_table`.\n",
    "3. **Sequence:** Standalone sequences in Oracle are replaced by `GENERATED ALWAYS AS IDENTITY` columns in Databricks Delta. The `ALTER_SEQUENCE` task is mapped to the `ALTER TABLE ... ALTER COLUMN ... RESTART WITH` syntax."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}